In [34]:
high_finesse_wavemeter_server.start_acquisition()

0

In [2]:
wlm = high_finesse_wavemeter_server._wavemeter

In [4]:
dll = wlm.dll

In [66]:
import ctypes
from ctypes import c_double, c_int, c_char_p, c_short, c_void_p, c_ulong,c_bool

In [30]:
# Example: Read pattern for index 0
pattern_index = 0

# Step 1: Get pattern item size and count
item_size = dll.GetPatternItemSize(pattern_index)
item_count = dll.GetPatternItemCount(pattern_index)
total_size = item_size * item_count

print(f"Item size: {item_size}, Item count: {item_count}, Total size: {total_size}")

# Step 2: Allocate array of c_short
PatternArrayType = c_short * total_size
pattern_array = PatternArrayType()


Item size: 2, Item count: 2048, Total size: 4096


In [ ]:
# Example: Read pattern for index 0
pattern_index = 0

# Step 1: Get pattern item size and count
item_size = dll.GetPatternItemSize(pattern_index)
item_count = dll.GetPatternItemCount(pattern_index)
total_size = item_size * item_count

print(f"Item size: {item_size}, Item count: {item_count}, Total size: {total_size}")

# Step 2: Allocate array of c_short
PatternArrayType = c_short * total_size
pattern_array = PatternArrayType()


In [9]:
# Step 1: Select pattern index
pattern_index = 0

# Step 2: Enable pattern
enable_result = dll.SetPattern(pattern_index, 1)
if enable_result != 0:
    raise RuntimeError(f"Failed to enable pattern at index {pattern_index}, error code: {enable_result}")
print(f"Pattern {pattern_index} enabled.")

# Step 3: Get pattern size and count
item_size = dll.GetPatternItemSize(pattern_index)
item_count = dll.GetPatternItemCount(pattern_index)
total_size = item_size * item_count
print(f"Pattern size: {item_size}, count: {item_count}, total entries: {total_size}")


Pattern 0 enabled.
Pattern size: 2, count: 2048, total entries: 4096


In [10]:

# Step 4: Allocate memory for data
PatternArrayType = c_short * total_size
pattern_array = PatternArrayType()

# Step 5: Read pattern data
read_result = dll.GetPatternData(pattern_index, pattern_array)
if read_result != 0:
    raise RuntimeError(f"Failed to read pattern data, error code: {read_result}")

# Step 6: Use pattern data
pattern_data = list(pattern_array)
print(f"Read pattern data (first 10 values): {pattern_data[:10]}")

ctypes.ArgumentError: argument 2: <class 'TypeError'>: Don't know how to convert parameter 2

========= Remote Traceback (1) =========
Traceback (most recent call last):
  File "C:\Users\tinPC\.conda\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 369, in _dispatch_request
    res = self._HANDLERS[handler](self, *args)
  File "C:\Users\tinPC\.conda\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 863, in _handle_call
    return obj(*args, **dict(kwargs))
ctypes.ArgumentError: argument 2: <class 'TypeError'>: Don't know how to convert parameter 2


In [ ]:

# Step 3: Call GetPatternData
result = dll.GetPatternData(pattern_index, pattern_array)
if result != 0:
    print(f"Failed to get pattern data. Error code: {result}")
else:
    # Convert to Python list
    pattern_data = list(pattern_array)
    print(f"Pattern data (length={len(pattern_data)}): {pattern_data[:10]}...")  # First 10 values

In [29]:
import numpy as np
from ctypes import c_double, c_long, POINTER, byref

def get_analysis_data(index: int, num_points: int = 1000):
    # Allocate an array for the output
    buffer = (c_double * num_points)()

    # Call the DLL function
    result = dll.GetAnalysisData(c_long(index), buffer)

    if result <= 0:
        raise RuntimeError("Failed to get analysis data or no data returned.")

    # Convert to numpy array
    data = np.ctypeslib.as_array(buffer)[:result]
    return data

In [13]:
get_analysis_data(0, 1000)

ctypes.ArgumentError: argument 1: <class 'TypeError'>: Don't know how to convert parameter 1

========= Remote Traceback (1) =========
Traceback (most recent call last):
  File "C:\Users\tinPC\.conda\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 369, in _dispatch_request
    res = self._HANDLERS[handler](self, *args)
  File "C:\Users\tinPC\.conda\envs\squdi\lib\site-packages\rpyc\core\protocol.py", line 863, in _handle_call
    return obj(*args, **dict(kwargs))
ctypes.ArgumentError: argument 1: <class 'TypeError'>: Don't know how to convert parameter 1


In [17]:
dll.GetAnalysisData.argtypes = [c_long, POINTER(c_double)]
dll.GetAnalysisData.restype = c_long

In [18]:
def get_analysis_data(dll, index: int, buffer_size: int = 4096):
    # Create a buffer of c_double
    buffer = (c_double * buffer_size)()

    # Call the function with a plain c_long and pointer to buffer
    result = dll.GetAnalysisData(c_long(index), buffer)

    if result <= 0:
        raise RuntimeError("Failed to get analysis data or no data returned.")

    # Convert to numpy array (only valid data points)
    return np.ctypeslib.as_array(buffer)[:result]

In [23]:
buffer = (c_double * 1000)()

In [27]:
# ULONG_PTR GetAnalysis(long Index)
dll.GetAnalysis.argtypes = [c_long]
dll.GetAnalysis.restype = POINTER(c_ulong)

NameError: name 'c_ulong' is not defined

In [57]:
dll.GetAnalysisData.argtypes = [c_long, POINTER(c_double)]
dll.GetAnalysisData.restype = c_long

In [60]:
result = dll.GetAnalysisData(c_long(4), buffer)

In [70]:
from ctypes import c_double, c_long, windll
import numpy as np
import matplotlib.pyplot as plt

# Load the DLL
dll = windll.LoadLibrary("wlmData.dll")

# Setup argtypes and restype
dll.GetAnalysisData.argtypes = [c_long, POINTER(c_double)]
dll.GetAnalysisData.restype = c_long

# Create a buffer and call the function
buffer_size = 4096
buffer = (c_double * buffer_size)()
result = dll.GetAnalysisData(c_long(4), buffer)

if result > 0:
    data = np.ctypeslib.as_array(buffer)[:result]
    plt.plot(data)
    plt.title("Channel 4 Analysis Data")
    plt.grid(True)
    plt.show()
else:
    print("Failed to get data.")

Failed to get data.


In [68]:
dll.SetAnalysisMode.argtypes = [c_bool]
dll.SetAnalysisMode.restype = c_long

In [71]:
dll.SetAnalysisMode(True)

-6

In [72]:
import numpy as np
import matplotlib.pyplot as plt
from ctypes import windll, c_bool, c_long, c_double, POINTER

# 1. Load the DLL
dll = windll.LoadLibrary("wlmData.dll")

# 2. Configure function signatures
dll.SetAnalysisMode.argtypes = [c_bool]
dll.SetAnalysisMode.restype = c_long

dll.GetAnalysisData.argtypes = [c_long, POINTER(c_double)]
dll.GetAnalysisData.restype = c_long

# 3. Enable analysis mode
res = dll.SetAnalysisMode(True)
if res == 0:
    raise RuntimeError("Failed to enable analysis mode")

# 4. Prepare buffer and fetch channel 4 data (index 3)
buffer_size = 4096
buffer = (c_double * buffer_size)()
count  = dll.GetAnalysisData(c_long(3), buffer)
if count <= 0:
    raise RuntimeError("No analysis data returned for channel 4")

# 5. Convert to NumPy and plot
data = np.ctypeslib.as_array(buffer)[:count]
plt.figure(figsize=(10, 6))
plt.plot(data, label="Channel 4 Analysis")
plt.title("HighFinesse Channel 4 Analysis Data")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude (a.u.)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

RuntimeError: No analysis data returned for channel 4

In [ ]:
data = np.ctypeslib.as_array(buffer)[:result]

In [ ]:
buffer.

In [56]:
channel_index = 3  # Channel 4 (indexing from 0)
data = get_analysis_data(dll, channel_index)

TypeError: an integer is required (got type WinDLL)

In [125]:
cCtrlStartMeasurment = ctypes.c_uint16(0x1002)
dll.Operation(cCtrlStartMeasurment)

0

In [109]:
# --- 1. Define Constants from the Manual ---
# Base Operation Constants are on page 124 of the manual.
cStop = 0x0000
# cCtrlStartRecord starts a measurement that also records to a file[cite: 1039].
cCtrlStartRecord = 0x0004

# --- 2. Load the DLL ---
# Use WinDLL for the 'stdcall' calling convention.
try:
    dll = ctypes.WinDLL("wlmData.dll")
    print("wlmData.dll loaded successfully.")
except OSError as e:
    print(f"Error loading wlmData.dll: {e}")
    exit()

# --- 3. Define Function Prototypes ---
# long Operation(unsigned short Op)
operation_func = dll.Operation
operation_func.restype = ctypes.c_long
operation_func.argtypes = [ctypes.c_uint16]

wlmData.dll loaded successfully.


In [110]:
# long SetOperationFile(char *Filename) [cite: 1046]
# This function tells the software the name of the file to use[cite: 1045].
set_operation_file_func = dll.SetOperationFile
set_operation_file_func.restype = ctypes.c_long
# Use ctypes.c_char_p for string pointers (char*).
set_operation_file_func.argtypes = [ctypes.c_char_p]

In [111]:
import os

In [156]:
# Define the full path for the recording file.
# IMPORTANT: Ensure the directory exists before running the script.
output_directory = r"C:\Users\tinPC\Desktop\test_wvm2"
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
    print(f"Created directory: {output_directory}")

filename = os.path.join(output_directory, "long_term_data4.ltr")
print(f"Data will be recorded to: {filename}")

Data will be recorded to: C:\Users\tinPC\Desktop\test_wvm2\long_term_data4.ltr


In [157]:
# C functions expect byte strings, so we encode the Python string.
encoded_filename = filename.encode('utf-8')

# Call SetOperationFile first to set the target file path.
print("Setting the operation file...")
result = set_operation_file_func(encoded_filename)

Setting the operation file...


In [131]:
import time

In [119]:
record_result = operation_func(cCtrlStartRecord)

In [158]:
# A return value of 0 means success.
if result == 0:
    print("Filename set successfully.")
    
    # Use a try...finally block to ensure recording is stopped.
    try:
        # Now, call Operation with cCtrlStartRecord to begin recording.
        print("Starting the recording...")
        record_result = operation_func(cCtrlStartRecord)
        
        if record_result == 0:
            print("Recording is active. Data is being saved to the file.")
            # Let the recording run for 10 seconds for this demo.
            time.sleep(10)
        else:
            print(f"Failed to start recording. Error code: {record_result}")
    
    finally:
        # Stop the measurement, which finalizes and closes the file.
        print("Stopping the recording...")
        stop_result = operation_func(cStop)
        if stop_result == 0:
            print(f"Recording stopped. File '{filename}' has been saved.")
        else:
            print(f"Failed to stop recording. Error code: {stop_result}")
else:
    # See page 112 in the manual for error code descriptions.
    print(f"Failed to set filename. Error code: {result}")

Filename set successfully.
Starting the recording...
Recording is active. Data is being saved to the file.
Stopping the recording...
Recording stopped. File 'C:\Users\tinPC\Desktop\test_wvm2\long_term_data4.ltr' has been saved.


In [140]:
record_result = operation_func(cCtrlStartRecord)

In [153]:
cCtrlStartMeasurment = ctypes.c_uint16(0x1002)
dll.Operation(cCtrlStartMeasurment)

0

In [154]:

dll.Operation(cStop)

0

In [124]:
stop_result = operation_func(cStop)

In [136]:
cCtrlOverwrite = 0x1000

In [155]:
# Combine Start and Overwrite flags using a bitwise OR
restart_command = cCtrlStartRecord | cCtrlOverwrite
operation_func(cCtrlOverwrite)

0

In [43]:
high_finesse_wavemeter_server._wavemeter.get_wavelength(channel=4)

484.12516919670816

In [73]:
dll.GetWavelengthNum.restype = ctypes.c_double
dll.GetWavelengthNum.argtypes = [ctypes.c_long, ctypes.c_double]

wavelength = dll.GetWavelengthNum(4, 0)

In [74]:
wavelength

619.2459076401157

In [76]:
dll.GetAnalysisData.argtypes = [c_long, POINTER(c_double)]
dll.GetAnalysisData.restype = c_long

In [91]:
string_buffer = ctypes.create_string_buffer(1024)
xp = ctypes.cast(string_buffer, ctypes.POINTER(ctypes.c_char))
dll.GetAnalysisData.argtypes = [c_long, xp]
dll.GetAnalysisData.restype = c_long
dat = dll.GetAnalysisData(3, string_buffer)

In [92]:
string_buffer.value  # This will give you the length of the string returned

b''

In [ ]:
dll.GetWavelengthNum.restype = ctypes.c_double
dll.GetWavelengthNum.argtypes = [ctypes.c_long, ctypes.c_double]

wavelength = dll.GetWavelengthNum(4, 0)

In [51]:
dll = ctypes.windll.LoadLibrary('wlmData.dll')

In [82]:
channel=4
string_buffer = ctypes.create_string_buffer(1024)
xp = ctypes.cast(string_buffer, ctypes.POINTER(ctypes.c_char))
dll.GetPIDCourseNum.restype = ctypes.c_long
dll.GetPIDCourseNum.argtypes = [ctypes.c_long, xp]
dll.GetPIDCourseNum(channel, string_buffer)

0

In [86]:
string_buffer.value.decode('utf-8')

'484.135'